# Quantize NLP models with Post-Training Quantization ​in NNCF
This tutorial demonstrates how to apply `INT8` quantization to the Natural Language Processing model known as [BERT](https://en.wikipedia.org/wiki/BERT_(language_model)), using the [Post-Training Quantization API](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/quantizing-models-post-training/basic-quantization-flow.html) (NNCF library). A fine-tuned [HuggingFace BERT](https://huggingface.co/transformers/model_doc/bert.html) [PyTorch](https://pytorch.org/) model, trained on the [Microsoft Research Paraphrase Corpus (MRPC)](https://www.microsoft.com/en-us/download/details.aspx?id=52398), will be used. The tutorial is designed to be extendable to custom models and datasets. It consists of the following steps:

- Download and prepare the BERT model and MRPC dataset.
- Define data loading and accuracy validation functionality.
- Prepare the model for quantization.
- Run optimization pipeline.
- Load and test quantized model.
- Compare the performance of the original, converted and quantized models.


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/language-quantize-bert/language-quantize-bert.ipynb" />



#### Table of contents:

- [Imports](#Imports)
- [Settings](#Settings)
- [Prepare the Model](#Prepare-the-Model)
- [Prepare the Dataset](#Prepare-the-Dataset)
- [Optimize model using NNCF Post-training Quantization API](#Optimize-model-using-NNCF-Post-training-Quantization-API)
- [Load and Test OpenVINO Model](#Load-and-Test-OpenVINO-Model)
    - [Select inference device](#Select-inference-device)
- [Compare F1-score of FP32 and INT8 models](#Compare-F1-score-of-FP32-and-INT8-models)
- [Compare Performance of the Original, Converted and Quantized Models](#Compare-Performance-of-the-Original,-Converted-and-Quantized-Models)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

In [1]:
%pip install -q "nncf>=2.5.0"
%pip install -q "torch>=2.8" transformers "datasets<4.0.0" "huggingface-hub==1.15.0" evaluate tqdm  --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "openvino>=2024.4.0"

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
optimum-onnx 0.1.0 requires transformers<4.58.0,>=4.36, but you have transformers 5.17.0 which is incompatible.
optimum-intel 1.27.0 requires transformers<4.58,>=4.45, but you have transformers 5.17.0 which is incompatible.


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


## Imports
[back to top ⬆️](#Table-of-contents:)


In [2]:
import time
from pathlib import Path
from zipfile import ZipFile
from typing import Iterable
from typing import Any

import datasets
import evaluate
import numpy as np
import nncf
from nncf.parameters import ModelType
import openvino as ov
import torch
from transformers import BertForSequenceClassification, BertTokenizer

# Fetch `notebook_utils` module
import requests

if not Path("notebook_utils.py").exists():
    r = requests.get(
        url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py",
    )

    open("notebook_utils.py", "w").write(r.text)
from notebook_utils import download_file, device_widget

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("language-quantize-bert.ipynb")

## Settings
[back to top ⬆️](#Table-of-contents:)


In [3]:
# Set the data and model directories, source URL and the filename of the model.
DATA_DIR = Path("data")
MODEL_DIR = Path("model")
MODEL_LINK = "https://download.pytorch.org/tutorial/MRPC.zip"
FILE_NAME = MODEL_LINK.split("/")[-1]
PRETRAINED_MODEL_DIR = MODEL_DIR / "MRPC"

DATA_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

## Prepare the Model
[back to top ⬆️](#Table-of-contents:)

Perform the following:

- Download and unpack pre-trained BERT model for MRPC by PyTorch.
- Convert the model to the OpenVINO Intermediate Representation (OpenVINO IR)

In [4]:
if not PRETRAINED_MODEL_DIR.exists():
    download_file(MODEL_LINK, directory=MODEL_DIR, show_progress=True)
    with ZipFile(f"{MODEL_DIR}/{FILE_NAME}", "r") as zip_ref:
        zip_ref.extractall(MODEL_DIR)

MRPC.zip:   0%|          | 0.00/387M [00:00<?, ?B/s]

Convert the original PyTorch model to the OpenVINO Intermediate Representation.

From OpenVINO 2023.0, we can directly convert a model from the PyTorch format to the OpenVINO IR format using model conversion API. Following PyTorch model formats are supported:

- `torch.nn.Module`
- `torch.jit.ScriptModule`
- `torch.jit.ScriptFunction`
- `torch.export.ExportedProgram`

In [5]:
MAX_SEQ_LENGTH = 128
input_shape = ov.PartialShape([1, -1])
ir_model_xml = Path(MODEL_DIR) / "bert_mrpc.xml"
core = ov.Core()

torch_model = BertForSequenceClassification.from_pretrained(PRETRAINED_MODEL_DIR)
torch_model.eval()

input_info = [
    ("input_ids", input_shape, np.int64),
    ("attention_mask", input_shape, np.int64),
    ("token_type_ids", input_shape, np.int64),
]
inputs = {
    "input_ids": torch.ones(1, MAX_SEQ_LENGTH, dtype=torch.int64),
    "attention_mask": torch.ones(1, MAX_SEQ_LENGTH, dtype=torch.int64),
    "token_type_ids": torch.ones(1, MAX_SEQ_LENGTH, dtype=torch.int64),
}

# Convert the PyTorch model to OpenVINO IR FP32.
if not ir_model_xml.exists():
    exported_model = torch.export.export(torch_model, args=(), kwargs=inputs)
    model = ov.convert_model(exported_model)
    # Assign input names expected by the rest of the notebook
    for idx, (name, _, _) in enumerate(input_info):
        model.inputs[idx].get_tensor().set_names({name})
    # Apply dynamic shapes by index to avoid name-lookup issues
    model.reshape({idx: shape for idx, (_, shape, _) in enumerate(input_info)})
    ov.save_model(model, str(ir_model_xml))
else:
    model = core.read_model(ir_model_xml)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

## Prepare the Dataset
[back to top ⬆️](#Table-of-contents:)

We download the [General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) dataset for the MRPC task from HuggingFace datasets.
Then, we tokenize the data with a pre-trained BERT tokenizer from HuggingFace.

In [6]:
def create_data_source():
    raw_dataset = datasets.load_dataset("glue", "mrpc", split="validation")
    tokenizer = BertTokenizer.from_pretrained(PRETRAINED_MODEL_DIR)

    def _preprocess_fn(examples):
        texts = (examples["sentence1"], examples["sentence2"])
        result = tokenizer(*texts, padding="max_length", max_length=MAX_SEQ_LENGTH, truncation=True)
        result["labels"] = examples["label"]
        return result

    processed_dataset = raw_dataset.map(_preprocess_fn, batched=True, batch_size=1)

    return processed_dataset


data_source = create_data_source()

README.md: 0.00B [00:00, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/308k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

## Optimize model using NNCF Post-training Quantization API
[back to top ⬆️](#Table-of-contents:)

[NNCF](https://github.com/openvinotoolkit/nncf) provides a suite of advanced algorithms for Neural Networks inference optimization in OpenVINO with minimal accuracy drop.
We will use 8-bit quantization in post-training mode (without the fine-tuning pipeline) to optimize BERT.

The optimization process contains the following steps:

1. Create a Dataset for quantization
2. Run `nncf.quantize` for getting an optimized model
3. Serialize OpenVINO IR model using `openvino.save_model` function

In [7]:
INPUT_NAMES = [key for key in inputs.keys()]


def transform_fn(data_item):
    """
    Extract the model's input from the data item.
    The data item here is the data item that is returned from the data source per iteration.
    This function should be passed when the data item cannot be used as model's input.
    """
    inputs = {name: np.asarray([data_item[name]], dtype=np.int64) for name in INPUT_NAMES}
    return inputs


calibration_dataset = nncf.Dataset(data_source, transform_fn)
# Quantize the model. By specifying model_type, we specify additional transformer patterns in the model.
quantized_model = nncf.quantize(model, calibration_dataset, model_type=ModelType.TRANSFORMER)

Output()

Output()

Output()

Output()

In [8]:
compressed_model_xml = Path(MODEL_DIR) / "quantized_bert_mrpc.xml"
ov.save_model(quantized_model, compressed_model_xml)

## Load and Test OpenVINO Model
[back to top ⬆️](#Table-of-contents:)

To load and test converted model, perform the following:

* Load the model and compile it for selected device.
* Prepare the input.
* Run the inference.
* Get the answer from the model output.

### Select inference device
[back to top ⬆️](#Table-of-contents:)

select device from dropdown list for running inference using OpenVINO

In [9]:
device = device_widget()

device

Dropdown(description='Device:', options=('CPU', 'GPU', 'AUTO'), value='CPU')

In [10]:
# Compile the model for a specific device.
compiled_quantized_model = core.compile_model(model=quantized_model, device_name=device.value)
output_layer = compiled_quantized_model.outputs[0]

The Data Source returns a pair of sentences (indicated by `sample_idx`) and the inference compares these sentences and outputs whether their meaning is the same. You can test other sentences by changing `sample_idx` to another value (from 0 to 407).

In [11]:
sample_idx = 5
sample = data_source[sample_idx]
inputs = {k: torch.unsqueeze(torch.tensor(sample[k]), 0) for k in ["input_ids", "token_type_ids", "attention_mask"]}

result = compiled_quantized_model(inputs)[output_layer]
result = np.argmax(result)

print(f"Text 1: {sample['sentence1']}")
print(f"Text 2: {sample['sentence2']}")
print(f"The same meaning: {'yes' if result == 1 else 'no'}")

Text 1: Wal-Mart said it would check all of its million-plus domestic workers to ensure they were legally employed .
Text 2: It has also said it would review all of its domestic employees more than 1 million to ensure they have legal status .
The same meaning: yes


## Compare F1-score of FP32 and INT8 models
[back to top ⬆️](#Table-of-contents:)


In [12]:
def validate(model: ov.Model, dataset: Iterable[Any]) -> float:
    """
    Evaluate the model on GLUE dataset.
    Returns F1 score metric.
    """
    compiled_model = core.compile_model(model, device_name=device.value)
    output_layer = compiled_model.output(0)

    metric = evaluate.load("glue", "mrpc")
    for batch in dataset:
        inputs = [np.expand_dims(np.asarray(batch[key], dtype=np.int64), 0) for key in INPUT_NAMES]
        outputs = compiled_model(inputs)[output_layer]
        predictions = outputs[0].argmax(axis=-1)
        metric.add_batch(predictions=[predictions], references=[batch["labels"]])
    metrics = metric.compute()
    f1_score = metrics["f1"]

    return f1_score


print("Checking the accuracy of the original model:")
metric = validate(model, data_source)
print(f"F1 score: {metric:.4f}")

print("Checking the accuracy of the quantized model:")
metric = validate(quantized_model, data_source)
print(f"F1 score: {metric:.4f}")

Checking the accuracy of the original model:


F1 score: 0.9019
Checking the accuracy of the quantized model:


F1 score: 0.9000


## Compare Performance of the Original, Converted and Quantized Models
[back to top ⬆️](#Table-of-contents:)

Compare the original PyTorch model with OpenVINO converted and quantized models (`FP32`, `INT8`) to see the difference in performance. It is expressed in Sentences Per Second (SPS) measure, which is the same as Frames Per Second (FPS) for images.

In [13]:
# Compile the model for a specific device.
compiled_model = core.compile_model(model=model, device_name=device.value)

In [14]:
num_samples = 50
sample = data_source[0]
inputs = {k: torch.unsqueeze(torch.tensor(sample[k]), 0) for k in ["input_ids", "token_type_ids", "attention_mask"]}

with torch.no_grad():
    start = time.perf_counter()
    for _ in range(num_samples):
        torch_model(torch.vstack(list(inputs.values())))
    end = time.perf_counter()
    time_torch = end - start
print(f"PyTorch model on CPU: {time_torch / num_samples:.3f} seconds per sentence, " f"SPS: {num_samples / time_torch:.2f}")

start = time.perf_counter()
for _ in range(num_samples):
    compiled_model(inputs)
end = time.perf_counter()
time_ir = end - start
print(f"IR FP32 model in OpenVINO Runtime/{device.value}: {time_ir / num_samples:.3f} " f"seconds per sentence, SPS: {num_samples / time_ir:.2f}")

start = time.perf_counter()
for _ in range(num_samples):
    compiled_quantized_model(inputs)
end = time.perf_counter()
time_ir = end - start
print(f"OpenVINO IR INT8 model in OpenVINO Runtime/{device.value}: {time_ir / num_samples:.3f} " f"seconds per sentence, SPS: {num_samples / time_ir:.2f}")

PyTorch model on CPU: 0.100 seconds per sentence, SPS: 10.02


IR FP32 model in OpenVINO Runtime/CPU: 0.012 seconds per sentence, SPS: 83.89


OpenVINO IR INT8 model in OpenVINO Runtime/CPU: 0.007 seconds per sentence, SPS: 139.77


Finally, measure the inference performance of OpenVINO `FP32` and `INT8` models. For this purpose, use [Benchmark Tool](https://docs.openvino.ai/2024/learn-openvino/openvino-samples/benchmark-tool.html) in OpenVINO.

> **Note**: The `benchmark_app` tool is able to measure the performance of the OpenVINO Intermediate Representation (OpenVINO IR) models only. For more accurate performance, run `benchmark_app` in a terminal/command prompt after closing other applications. Run `benchmark_app -m model.xml -d CPU` to benchmark async inference on CPU for one minute. Change `CPU` to `GPU` to benchmark on GPU. Run `benchmark_app --help` to see an overview of all command-line options.

In [15]:
# Inference FP32 model (OpenVINO IR)
!benchmark_app -m $ir_model_xml -shape [1,128],[1,128],[1,128] -d {device.value} -api sync

[Step 1/11] Parsing and validating input arguments
[ INFO ] Parsing input parameters
[Step 2/11] Loading OpenVINO Runtime
[ INFO ] OpenVINO:
[ INFO ] Build ................................. 2026.4.1-22982-e213a147257-releases/2026/4
[ INFO ] 
[ INFO ] Device info:
[ INFO ] CPU
[ INFO ] Build ................................. 2026.4.1-22982-e213a147257-releases/2026/4
[ INFO ] 
[ INFO ] 
[Step 3/11] Setting device configuration
[ WARNING ] Performance hint was not explicitly specified in command line or config file. Device(CPU) performance hint will be set to PerformanceMode.LATENCY.
[Step 4/11] Reading model files
[ INFO ] Loading model files


[ INFO ] Read model took 9.53 ms
[ INFO ] Original model I/O parameters:
[ INFO ] Model inputs:
[ INFO ]     input_ids (node: input_ids) : i64 / [...] / [1,?]
[ INFO ]     attention_mask (node: attention_mask) : i64 / [...] / [1,?]
[ INFO ]     token_type_ids (node: token_type_ids) : i64 / [...] / [1,?]
[ INFO ] Model outputs:
[ INFO ]     linear_73:0 (node: linear_73/aten.linear.default/Add) : f32 / [...] / [1,2]
[Step 5/11] Resizing model to match image sizes and given batch
[ INFO ] Model batch size: 1
[ INFO ] Reshaping model: 'input_ids': [1,128], 'attention_mask': [1,128], 'token_type_ids': [1,128]
[ INFO ] Reshape model took 3.89 ms
[Step 6/11] Configuring input of the model
[ INFO ] Model inputs:
[ INFO ]     input_ids (node: input_ids) : i64 / [...] / [1,128]
[ INFO ]     attention_mask (node: attention_mask) : i64 / [...] / [1,128]
[ INFO ]     token_type_ids (node: token_type_ids) : i64 / [...] / [1,128]
[ INFO ] Model outputs:
[ INFO ]     linear_73:0 (node: linear_73/aten.

[ INFO ] Compile model took 350.27 ms
[ INFO ] Start of compilation memory usage: Peak 1638896 KB
[ INFO ] End of compilation memory usage: Peak 2943664 KB
[ INFO ] Compile model ram used 1304768 KB
[Step 8/11] Querying optimal runtime parameters
[ INFO ] Model:
[ INFO ]   NETWORK_NAME: Model0
[ INFO ]   OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
[ INFO ]   NUM_STREAMS: 1
[ INFO ]   INFERENCE_NUM_THREADS: 16
[ INFO ]   PERF_COUNT: NO
[ INFO ]   INFERENCE_PRECISION_HINT: <Type: 'bfloat16'>
[ INFO ]   PERFORMANCE_HINT: LATENCY
[ INFO ]   EXECUTION_MODE_HINT: ExecutionMode.PERFORMANCE
[ INFO ]   PERFORMANCE_HINT_NUM_REQUESTS: 0
[ INFO ]   ENABLE_CPU_PINNING: True
[ INFO ]   ENABLE_CPU_RESERVATION: False
[ INFO ]   SCHEDULING_CORE_TYPE: SchedulingCoreType.ANY_CORE
[ INFO ]   MODEL_DISTRIBUTION_POLICY: set()
[ INFO ]   ENABLE_HYPER_THREADING: False
[ INFO ]   EXECUTION_DEVICES: ['CPU']
[ INFO ]   CPU_DENORMALS_OPTIMIZATION: False
[ INFO ]   LOG_LEVEL: Level.NO
[ INFO ]   CPU_SPARSE_WEIGHTS_DECOMPR

[Step 11/11] Dumping statistics report
[ INFO ] Execution Devices:['CPU']
[ INFO ] Count:            5271 iterations
[ INFO ] Duration:         60008.95 ms
[ INFO ] Latency:
[ INFO ]    Median:        11.27 ms
[ INFO ]    Average:       11.30 ms
[ INFO ]    Min:           10.62 ms
[ INFO ]    Max:           12.66 ms
[ INFO ] Throughput:   87.84 FPS


In [16]:
# Inference INT8 model (OpenVINO IR)
! benchmark_app -m $compressed_model_xml -shape [1,128],[1,128],[1,128] -d {device.value} -api sync

[Step 1/11] Parsing and validating input arguments
[ INFO ] Parsing input parameters
[Step 2/11] Loading OpenVINO Runtime
[ INFO ] OpenVINO:
[ INFO ] Build ................................. 2026.4.1-22982-e213a147257-releases/2026/4
[ INFO ] 
[ INFO ] Device info:
[ INFO ] CPU
[ INFO ] Build ................................. 2026.4.1-22982-e213a147257-releases/2026/4
[ INFO ] 
[ INFO ] 
[Step 3/11] Setting device configuration
[ WARNING ] Performance hint was not explicitly specified in command line or config file. Device(CPU) performance hint will be set to PerformanceMode.LATENCY.
[Step 4/11] Reading model files
[ INFO ] Loading model files
[ INFO ] Read model took 18.91 ms
[ INFO ] Original model I/O parameters:
[ INFO ] Model inputs:
[ INFO ]     input_ids (node: input_ids) : i64 / [...] / [1,?]
[ INFO ]     attention_mask (node: attention_mask) : i64 / [...] / [1,?]
[ INFO ]     token_type_ids (node: token_type_ids) : i64 / [...] / [1,?]
[ INFO ] Model outputs:
[ INFO ]     linear

[ INFO ] Model inputs:
[ INFO ]     input_ids (node: input_ids) : i64 / [...] / [1,128]
[ INFO ]     attention_mask (node: attention_mask) : i64 / [...] / [1,128]
[ INFO ]     token_type_ids (node: token_type_ids) : i64 / [...] / [1,128]
[ INFO ] Model outputs:
[ INFO ]     linear_73:0 (node: linear_73/aten.linear.default/Add) : f32 / [...] / [1,2]
[Step 7/11] Loading the model to the device


[ INFO ] Compile model took 583.01 ms
[ INFO ] Start of compilation memory usage: Peak 1571308 KB
[ INFO ] End of compilation memory usage: Peak 3268588 KB
[ INFO ] Compile model ram used 1697280 KB
[Step 8/11] Querying optimal runtime parameters
[ INFO ] Model:
[ INFO ]   NETWORK_NAME: Model0
[ INFO ]   OPTIMAL_NUMBER_OF_INFER_REQUESTS: 1
[ INFO ]   NUM_STREAMS: 1
[ INFO ]   INFERENCE_NUM_THREADS: 16
[ INFO ]   PERF_COUNT: NO
[ INFO ]   INFERENCE_PRECISION_HINT: <Type: 'bfloat16'>
[ INFO ]   PERFORMANCE_HINT: LATENCY
[ INFO ]   EXECUTION_MODE_HINT: ExecutionMode.PERFORMANCE
[ INFO ]   PERFORMANCE_HINT_NUM_REQUESTS: 0
[ INFO ]   ENABLE_CPU_PINNING: True
[ INFO ]   ENABLE_CPU_RESERVATION: False
[ INFO ]   SCHEDULING_CORE_TYPE: SchedulingCoreType.ANY_CORE
[ INFO ]   MODEL_DISTRIBUTION_POLICY: set()
[ INFO ]   ENABLE_HYPER_THREADING: False
[ INFO ]   EXECUTION_DEVICES: ['CPU']
[ INFO ]   CPU_DENORMALS_OPTIMIZATION: False
[ INFO ]   LOG_LEVEL: Level.NO
[ INFO ]   CPU_SPARSE_WEIGHTS_DECOMPR

[ INFO ] First inference took 10.37 ms


[Step 11/11] Dumping statistics report
[ INFO ] Execution Devices:['CPU']
[ INFO ] Count:            8855 iterations
[ INFO ] Duration:         60004.21 ms
[ INFO ] Latency:
[ INFO ]    Median:        6.70 ms
[ INFO ]    Average:       6.71 ms
[ INFO ]    Min:           6.33 ms
[ INFO ]    Max:           7.70 ms
[ INFO ] Throughput:   147.57 FPS
